# Day 1 — Introduction & URSim Setup

Welcome to Day 1 of the **UR5 AI Workshop**!

### 🎯 Learning Objectives
By the end of this notebook, you will:
1. Understand the UR5 robot’s architecture (joints, frames, communication).
2. Set up and run the URSim virtual robot.
3. Connect to URSim via Python using `ur_rtde`.
4. Send simple motion commands and read robot state data.


## 1. URSim Overview
URSim is a simulation environment provided by Universal Robots. It replicates the behavior of UR5/UR10 robots.

**Key features:**
- Test URScript programs without a physical robot.
- Network communication identical to real hardware.
- Useful for AI and control development.

**Common ports:**
- 30001–30004: URScript / dashboard communication.
- 30002: secondary client interface.
- 30004: RTDE interface (Real-Time Data Exchange).


## 🧩 Exercise 1 — URSim Installation and Configuration
**Goal:** Install URSim and verify that it runs locally.

**Steps:**
1. Install **URSim** (either standalone or in VirtualBox).
2. Configure network (use host-only adapter if in VirtualBox).
3. Launch URSim → *Polyscope GUI should appear*.
4. Set the robot to *Remote Control mode*.

**Deliverable:** Screenshot showing URSim running successfully.

## 🧩 Exercise 2 — Connect to URSim via Python

In [ ]:
# Student Task
from rtde_control import RTDEControlInterface
from rtde_receive import RTDEReceiveInterface

ROBOT_IP = '127.0.0.1'  # Replace if needed
rtde_c = RTDEControlInterface(ROBOT_IP)
rtde_r = RTDEReceiveInterface(ROBOT_IP)

joint_positions = rtde_r.getActualQ()
print('Current joint positions:', joint_positions)

## 🧩 Exercise 3 — Command a Simple Movement

In [ ]:
# Student Task
import time
target_q = [0, -1.57, 1.57, 0, 1.57, 0]
rtde_c.moveJ(target_q, 0.5, 0.5)
time.sleep(2)
print('Reached target position')

## 🧩 Exercise 4 — Safety and Error Handling

In [ ]:
# Student Task
import math

JOINT_LIMIT = math.pi   # max ±2π
MAX_SPEED = 1.0         # rad/s   (controller allows 3.34)
MAX_ACCEL = 1.4         # rad/s²  (conservative workshop limit)
TOLERANCE = 1e-3        # rad, accepted final joint error

def safe_move(target_q, speed=0.3, accel=0.5):
    if not (0 < speed <= MAX_SPEED):
        print(f"Speed must be between 0 and {MAX_SPEED}, got {speed}")
        return False
    if not (0 < accel <= MAX_ACCEL):
        print(f"Acceleration must be between 0 and {MAX_ACCEL}, got {accel}")
        return False

    if len(target_q) != 6:
        print(f"Target must have 6 joint values, got {len(target_q)}")
        return False

    if rtde_r.isProtectiveStopped() or rtde_r.isEmergencyStopped():
        print("Robot is protective/emergency stopped")
        return False
    if rtde_r.getRobotMode() != 7 or rtde_r.getSafetyMode() != 1:   # 7 = RUNNING, 1 = NORMAL
        print(f"Robot not ready (robot mode {rtde_r.getRobotMode()}, safety mode {rtde_r.getSafetyMode()})")
        return False

    current_q = rtde_r.getActualQ()
    print('Current joint positions:', current_q)

    for i in range(len(target_q)):
        if not (-JOINT_LIMIT <= target_q[i] <= JOINT_LIMIT):
            print(f"Target joint {i} position {target_q[i]} is out of safe limits.")
            return False

    ok = rtde_c.moveJ(target_q, speed, accel)   # blocks until the move is finished
    error = max(abs(a - t) for a, t in zip(rtde_r.getActualQ(), target_q))
    if not ok or error > TOLERANCE:
        print(f"Move failed (moveJ returned {ok}, joint error {error:.4f} rad)")
        return False

    print('Reached target position safely')
    return True

## 🎓 Challenge — Log and Plot Robot Motion
**Goal:** Log the robot’s TCP position while moving and visualize the trajectory.

Hints:
- Use `getActualTCPPose()` inside a loop.
- Store data in a list, then plot with matplotlib.
- Save to `trajectory.csv` for later analysis.

In [ ]:
# Student Task
import threading
import time
import matplotlib.pyplot as plt
import polars as pl

SAMPLE_PERIOD = 0.02    # s  → 50 Hz logging
COLUMNS = ['t', 'x', 'y', 'z', 'rx', 'ry', 'rz']

def log_tcp(samples, stop_event):
    """Record (t, x, y, z, rx, ry, rz) every SAMPLE_PERIOD until stop_event is set."""
    t0 = time.time()
    while not stop_event.is_set():
        samples.append([time.time() - t0] + rtde_r.getActualTCPPose())
        time.sleep(SAMPLE_PERIOD)

waypoints = [
    [0, -1.57, 1.57, 0, 1.57, 0],
    [1.0, -1.2, 1.2, -1.57, -1.57, 0],
    [-1.0, -1.8, 1.8, -1.57, -1.57, 0],
    [0, -1.57, 1.57, 0, 1.57, 0],
]

samples = []
stop_event = threading.Event()
logger = threading.Thread(target=log_tcp, args=(samples, stop_event))
logger.start()
try:
    for q in waypoints:
        if not safe_move(q, speed=0.5):
            print('Stopping: waypoint refused or failed', q)
            break
finally:
    stop_event.set()       
    logger.join()

df = pl.DataFrame(samples, schema=COLUMNS, orient='row')
print(f'Logged {df.height} samples over {df["t"].max():.1f} s')
print(df.select('x', 'y', 'z').describe())

df.write_csv('trajectory.csv')
print('Saved trajectory.csv')

t, x, y, z = (df[c].to_numpy() for c in ('t', 'x', 'y', 'z'))
fig = plt.figure(figsize=(12, 5))

ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.plot(x, y, z)
ax.scatter(x[0], y[0], z[0], color='green', label='start')
ax.scatter(x[-1], y[-1], z[-1], color='red', label='end')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]'); ax.set_zlabel('z [m]')
ax.set_title('TCP trajectory (base frame)')
ax.legend()

ax2 = fig.add_subplot(1, 2, 2)
for values, label in ((x, 'x'), (y, 'y'), (z, 'z')):
    ax2.plot(t, values, label=label)
ax2.set_xlabel('time [s]'); ax2.set_ylabel('position [m]')
ax2.set_title('TCP position over time')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.show()